# Open-XTTS-Trainer on Colab (run 5: locked environment, identity canary, multilingual baseline)

Runs the parts of docs/recipe_design.md (§6) that need the XTTS-v2 weights and a GPU, on code cloned from GitHub at
one exact commit, with exactly the dependency versions in `uv.lock`.

| Step | Phase | What it proves |
|---|---|---|
| 1–3b | 0 | Clone at a commit SHA; install exactly `uv.lock`'s versions and verify every one; record commit, lock hash, versions and code fingerprint |
| 4 | 0 | Every upstream artifact matches `artifacts.lock.json`; `mel_stats` three-way check |
| 5 | 0 | Full test suite passes in this environment |
| 6 | 1 | Reference voices rebuilt from CMU ARCTIC and verified against the committed lock |
| 7 | 5 | Model-gated tests, results saved: **identity canary on the released weights**, vocoder round trip, real `dvae.pth` strict load, upstream weights in our models, real-vocab `[UNK]` check |
| 7b | 6 | Real forced aligner on JFK (±1 s cut shifts rejected); sample set prepared from **SHA-256-pinned** downloads with zero edits |
| 7c | 7 | Stage 2 smoke: strict load, a few steps, export, strict reload, synthesis. Wiring only: 6 clips say nothing about quality |
| 7d | 7 | Stage 3 smoke, 30 steps, logging the fp16 loss scale and skipped steps (run 4's first steps were flat). Shows the decoder-training path works, not that inference improves |
| 7e | 7 | *(off: recorded in run 4)* Stage 1, normal and frozen codebook |
| 8 | 1 | **Stock baseline, all 5 eval languages** at pilot scale (12 sentences × 5 voices × 3 seeds per language), one language at a time; resumed pieces are checked against the expected inputs; one index over all five |
| 9 | — | Evidence bundle to bring back into the repo |

Every shell step goes through `run()`, which fails the cell when the command fails (IPython's `!cmd` doesn't). The
XTTS weights are re-hashed after every step that uses them, so a step that changes them fails on the spot.

**Licenses you accept by running this:** XTTS-v2 weights are under the Coqui Public Model License (non-commercial only).
ECAPA2 and MMS_FA are CC BY-NC 4.0. CMU ARCTIC allows any use if its notice is kept, modifications are marked and the
authors are credited; step 6 saves each speaker's `COPYING` and a `MODIFICATIONS.txt` next to the clips.

**Runtime:** GPU (T4 or better), about 1.5–2 h on a T4: roughly 40–50 min for steps 1–7d and about an hour for step 8.
Step 3 restarts the runtime once, by design: continue with step 3b. If the session disconnects during step 8, rerun
steps 1, 3b, 4 and 6, then step 8: it reuses the languages already on Drive after checking them. Every step fails
loudly; if a cell errors, stop and bring the output back. Don't edit around it.

## 1. GPU and Drive
Results go to Google Drive so a disconnected session doesn't lose them.

In [ ]:
!nvidia-smi
import json, os
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/open_xtts_run5'
os.makedirs(RESULTS, exist_ok=True)
print('Results ->', RESULTS)

## 2. Get the code from GitHub at one commit
`GIT_REF` must be a full 40-character commit SHA, so the run is tied to exact code by construction. Get it with
`git rev-parse HEAD` on the commit you want (a notebook can't carry its own commit's SHA, so there is no default).

In [ ]:
import re, subprocess
REPO_URL = 'https://github.com/amitbhakt/open-xtts-trainer.git'
GIT_REF = ''  # full commit SHA, e.g. the output of `git rev-parse HEAD`
assert re.fullmatch(r'[0-9a-f]{40}', GIT_REF), 'set GIT_REF to a full 40-character commit SHA'

%cd /content
!rm -rf open-xtts-trainer
subprocess.run(['git', 'clone', '-q', REPO_URL, 'open-xtts-trainer'], check=True)
%cd /content/open-xtts-trainer
subprocess.run(['git', '-c', 'advice.detachedHead=false', 'checkout', '-q', GIT_REF], check=True)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert COMMIT == GIT_REF, (COMMIT, GIT_REF)
assert not subprocess.check_output(['git', 'status', '--porcelain'], text=True).strip(), 'fresh clone is not clean'
json.dump({'repo': REPO_URL, 'commit': COMMIT, 'results': RESULTS}, open('/content/run_state.json', 'w'))
print(subprocess.check_output(['git', 'log', '-1', '--format=%H %ad %s', '--date=iso'], text=True))

## 3. Install exactly the locked dependencies, then restart
`uv.lock` is exported and installed with `--no-deps`, so nothing is re-resolved: Colab's preinstalled torch is
replaced by the locked one (runs 1–4 used Colab's torch 2.11, not the lock's). `torchcodec` comes from PyTorch's CPU
index, so it is installed on its own; adding that index for everything could swap torch for a CPU-only build.
**The runtime then restarts, by design:** packages replaced under a running kernel can't be trusted. Continue with step 3b.

In [ ]:
import subprocess
def sh(cmd):
    subprocess.run(['bash', '-o', 'pipefail', '-c', cmd], check=True)
sh('pip install -q uv==0.12.13')
sh('uv export --frozen --no-hashes --no-emit-project --extra dev --extra eval -o /content/requirements.lock.txt')
# torchcodec has one line per platform; keep the one whose marker holds here (by evaluating it, not by text)
from packaging.requirements import Requirement
lines = [l.strip() for l in open('/content/requirements.lock.txt') if l.strip() and not l.lstrip().startswith('#')]
codec = [l for l in lines if Requirement(l).name == 'torchcodec' and Requirement(l).marker.evaluate()]
assert len(codec) == 1, codec
open('/content/requirements.main.txt', 'w').write('\n'.join(l for l in lines if Requirement(l).name != 'torchcodec') + '\n')
open('/content/requirements.torchcodec.txt', 'w').write(codec[0].split(';')[0].strip() + '\n')
print('torchcodec:', open('/content/requirements.torchcodec.txt').read().strip())
sh('uv pip install --system --break-system-packages --no-deps -r /content/requirements.main.txt')
sh('uv pip install --system --break-system-packages --no-deps --index-url https://download.pytorch.org/whl/cpu '
   '-r /content/requirements.torchcodec.txt')
sh('uv pip install --system --break-system-packages --no-deps -e /content/open-xtts-trainer')
sh('pip uninstall -y -q torchvision')  # Colab preinstalls torchvision; orphaned version breaks transformers
print('Installed. Restarting the runtime; continue with step 3b.')
import os; os.kill(os.getpid(), 9)


## 3b. After the restart: reload state, verify every locked version, record the environment
Defines `run()`, which every later step uses: it runs a shell command with `pipefail` and fails the cell if the
command fails. `check_artifacts()` re-hashes the XTTS weights against `artifacts.lock.json`.

In [ ]:
import hashlib, json, os, platform, subprocess, sys
state = json.load(open('/content/run_state.json'))
COMMIT, RESULTS = state['commit'], state['results']
os.chdir('/content/open-xtts-trainer'); sys.path.insert(0, '.')
assert os.path.isdir(RESULTS), f'{RESULTS} missing: Drive is not mounted; rerun step 1'
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == COMMIT

def run(cmd, tail=None):
    """Run a shell command with pipefail; raise if it fails. tail=N prints only the last N lines (80 on failure)."""
    p = subprocess.Popen(['bash', '-o', 'pipefail', '-c', cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if tail is None:
            print(line, end='')
    p.wait()
    if tail is not None:
        print(''.join(lines[-(tail if p.returncode == 0 else 80):]), end='')
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)
    return ''.join(lines)

ARTIFACTS = json.load(open('artifacts.lock.json'))
def check_artifacts(after):
    """Every XTTS artifact in xtts_base/ still matches its lock entry; names the step that broke it otherwise."""
    for name, entry in ARTIFACTS.items():
        path = os.path.join('xtts_base', name)
        assert os.path.exists(path), f'{path} missing during/after {after}'
        digest = hashlib.sha256()
        with open(path, 'rb') as f:
            for chunk in iter(lambda: f.read(1 << 24), b''):
                digest.update(chunk)
        assert digest.hexdigest() == entry['sha256'], f'{path} changed during {after}: {digest.hexdigest()}'
    print(f'artifacts unchanged after {after}')

# Every locked package is installed at exactly its locked version (the git dependency at its locked commit)
from importlib.metadata import distribution, version
from packaging.requirements import Requirement
mismatched = []
for line in open('/content/requirements.lock.txt'):
    line = line.strip()
    if not line or line.startswith('#'):
        continue
    req = Requirement(line)
    if req.marker and not req.marker.evaluate():
        continue
    if req.url:
        commit = json.loads(distribution(req.name).read_text('direct_url.json'))['vcs_info']['commit_id']
        if not req.url.endswith('@' + commit):
            mismatched.append((req.name, req.url, commit))
    elif version(req.name) not in req.specifier:
        mismatched.append((req.name, str(req.specifier), version(req.name)))
assert not mismatched, mismatched
print('every locked package installed at its locked version')

import torch, torchaudio, transformers
from eval.harness import get_code_fingerprint
env = {
    'git': {'repo': state['repo'], 'commit': COMMIT},
    'uv_lock_sha256': hashlib.sha256(open('uv.lock', 'rb').read()).hexdigest(),
    'python': platform.python_version(),
    'torch': torch.__version__, 'torchaudio': torchaudio.__version__, 'transformers': transformers.__version__,
    'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'code_fingerprint': get_code_fingerprint(),
    'pip_freeze': subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True).splitlines(),
}
assert env['gpu'], 'no GPU: set Runtime > Change runtime type > T4 GPU'
os.makedirs('docs/evidence', exist_ok=True)
json.dump(env, open('docs/evidence/environment_colab.json', 'w'), indent=2)
print({k: v for k, v in env.items() if k != 'pip_freeze'})

## 4. Fetch and verify every upstream artifact; three-way `mel_stats` check (Phase 0)
Downloads about 2.1 GB into `xtts_base/`. Fails on any hash, size or `mel_stats` mismatch. The report goes to
`docs/evidence/phase0/`, not over the committed `docs/evidence/artifact_verification.json`: changing a tracked
file would mark every manifest of this run dirty.

In [ ]:
os.makedirs('docs/evidence/phase0', exist_ok=True)
run('python scripts/fetch_base_artifacts.py --dest xtts_base --check_mel_stats --report docs/evidence/phase0/artifact_verification.json')
report = json.load(open('docs/evidence/phase0/artifact_verification.json'))
assert all(a['sha256_ok'] and a['size_ok'] for a in report['artifacts'].values()), 'artifact mismatch'
assert report['mel_stats_check']['complete'] and report['mel_stats_check']['all_equal'], report['mel_stats_check']
check_artifacts('step 4')

## 5. Full test suite in this environment (Phase 0)
Takes 3–8 minutes. Model-gated tests that need step 6's voices are skipped here and run in step 7.

In [ ]:
run('python -m pytest -q -p no:cacheprovider', tail=8)
check_artifacts('step 5')

## 6. Rebuild and verify the reference voices (Phase 1)
Downloads 5 CMU ARCTIC tarballs (about 400 MB) plus each speaker's license. The clips and licenses must match the
committed lock (`eval/dataset/reference_voices.lock.json`).

In [ ]:
run('python scripts/fetch_reference_voices.py', tail=10)  # verifies against the committed lock
for sid, e in json.load(open('eval/dataset/reference_voices.lock.json'))['voices'].items():
    print(f"{sid}: {e['duration_s']}s  {e['utterances_used']}")

## 7. Model-gated tests (Phase 5)
The identity canary on the released weights (the released model through our export path must come back
bit-identical), the vocoder round-trip canary, the real `dvae.pth` strict load, upstream weights loading into our
factory models, and real-vocab `[UNK]` detection. Results are saved as JUnit XML; the cell fails if any of them
failed or was skipped.

In [ ]:
import xml.etree.ElementTree as ET
os.makedirs('docs/evidence/phase5', exist_ok=True)
xml_path = 'docs/evidence/phase5/model_gated_tests.xml'
try:
    run('python -m pytest -v -rs -p no:cacheprovider tests/test_invariants.py tests/test_spec_parity.py tests/test_data_contract.py '
        '-k "round_trip or upstream_checkpoint_loads or upstream_dvae_loads or real_vocab or identity_canary" '
        f'--junitxml {xml_path}', tail=14)
finally:
    if os.path.exists(xml_path):
        outcome = {c.get('name'): next((x.tag for x in c if x.tag in ('failure', 'error', 'skipped')), 'passed')
                   for c in ET.parse(xml_path).getroot().iter('testcase')}
        print(json.dumps(outcome, indent=1))
assert outcome.get('test_identity_canary_released_model') == 'passed', 'identity canary on released weights did not pass'
assert all(v == 'passed' for v in outcome.values()), 'a model-gated test was skipped'
check_artifacts('step 7')

## 7b. Phase 6: forced alignment with the real aligner
Downloads MMS_FA (1.26 GB, CC BY-NC 4.0). Then:
- JFK is aligned and cut at pauses; the clips must re-align cleanly, and a cut moved by ±1 s must be rejected;
- the sample set (LJSpeech + JFK) is prepared and must pass validation with zero manual edits. Every download is
  checked against its SHA-256 pin, and the pins used are saved with the evidence.

In [ ]:
run('python -c "import torchaudio; torchaudio.pipelines.MMS_FA.get_model()"', tail=2)
run('python -m pytest -v -p no:cacheprovider tests/test_prepare.py -k mms', tail=6)
run('rm -rf sample_audio && python -m xtts_trainer.speech_data --source sample --output_dir sample_audio --vocab xtts_base/vocab.json', tail=12)
run('python -m xtts_trainer.data.validate sample_audio/dataset.jsonl --vocab xtts_base/vocab.json --realign '
    '--report sample_audio/realign_report.json', tail=3)
os.makedirs('docs/evidence/phase6', exist_ok=True)
from xtts_trainer.speech_data import SOURCE_SETS, SOURCES
pins = {name: SOURCES[name].sha256 for name in SOURCE_SETS['sample']}
json.dump(pins, open('docs/evidence/phase6/source_pins_used.json', 'w'), indent=2)  # each verified on download
run('cp sample_audio/prep_report.json sample_audio/validation_report.json sample_audio/realign_report.json '
    'sample_audio/dataset.jsonl docs/evidence/phase6/')
for r in json.load(open('sample_audio/prep_report.json'))['recordings']:
    print(r['input'], {k: r.get(k) for k in ('segments', 'kept_fraction', 'alignment_score')}, 'dropped:', r.get('dropped'))

## 7c. Phase 7: Stage 2 smoke run through upstream GPTTrainer, real weights
3 epochs on the sample set (6 train, 2 eval clips) from the released `model.pth`, loaded strictly; export, strict
reload into stock `Xtts`, one synthesized sentence. This checks wiring, strict loading and export only: six clips at
batch 2 say nothing about training quality. About 7 GB of GPU memory and a few minutes on a T4.

In [ ]:
assert os.path.exists('sample_audio/dataset.jsonl'), 'step 7b did not produce the sample set'
os.makedirs('docs/evidence/phase7', exist_ok=True)
try:
    run('python -m xtts_trainer.stage2_gpt.train --dataset sample_audio/dataset.jsonl --base_dir xtts_base '
        '--output_dir runs/gpt_smoke --epochs 3 --batch_size 2 --grad_accum_steps 1 --eval_fraction 0.25 '
        '> runs_gpt_smoke.log 2>&1')
finally:
    run('cp runs_gpt_smoke.log docs/evidence/phase7/runs_gpt_smoke_log.txt && tail -25 runs_gpt_smoke.log')
summary = json.load(open('runs/gpt_smoke/export/training_summary.json'))
print(json.dumps({k: summary[k] for k in ('eval_loss_before', 'eval_loss_best', 'improved', 'best_step', 'train_clips', 'eval_clips')}, indent=1))
import torch, soundfile as sf
from xtts_trainer.stage2_gpt.export import load_for_inference
xtts = load_for_inference('runs/gpt_smoke/export', 'cuda')
lat, spk = xtts.get_conditioning_latents(audio_path=['eval/reference_voices/arctic_slt.wav'])
torch.manual_seed(42)
wav = xtts.inference('The quick brown fox jumps over the lazy dog.', 'en', lat, spk, temperature=0.7)['wav']
sf.write('docs/evidence/phase7/stage2_smoke_sample.wav', wav, 24000)
run('cp runs/gpt_smoke/export/training_summary.json docs/evidence/phase7/')
print(f'synthesized {len(wav) / 24000:.2f}s')
check_artifacts('step 7c')

## 7d. Phase 7: Stage 3 smoke run on real latents (chained from Stage 2)
Extracts latents with the Stage 2 export (eval-mode GPT), then fine-tunes the decoder for 30 steps: 10 of
discriminator warm-up, then adversarial. Run 4 already ran 120 steps; this run only needs the early steps, where
each validation record now carries the fp16 loss scale (`grad_scale_g`) and the number of generator steps the
scaler skipped (`skipped_steps_g`). This shows whether skipped steps explain run 4's flat start. A better validation
mel loss shows the decoder-training path works; it says nothing about inference until a decoder is packaged with a
Stage 2 export and measured end to end.

In [ ]:
import gc
for name in ('xtts', 'lat', 'spk', 'wav'):
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

from xtts_trainer.config import HiFiGANConfig
from xtts_trainer.stage3_hifigan.extract_latents import extract_latents
from xtts_trainer.stage3_hifigan.train import train_hifigan
extract_latents('sample_audio/dataset.jsonl', 'runs/latents', checkpoint_dir='runs/gpt_smoke/export', device='cuda')
cfg3 = HiFiGANConfig(batch_size=4, epochs=15, disc_warmup_steps=10, val_every_steps=2, sample_every_steps=10)
best3 = train_hifigan('sample_audio/dataset.jsonl', 'runs/latents', 'runs/hifigan_smoke', cfg3,
                      checkpoint_dir='runs/gpt_smoke/export', device='cuda')
ck3 = torch.load(best3, map_location='cpu', weights_only=False)
print({k: ck3[k] for k in ('val_mel_l1', 'reference_val_mel_l1', 'beats_start', 'step')})
for m in map(json.loads, open('runs/hifigan_smoke/metrics.jsonl')):
    print('step', m['step'], 'mel L1', round(m['val_mel_l1'], 4), 'scale', m['grad_scale_g'], 'skipped', m['skipped_steps_g'])
os.makedirs('docs/evidence/phase7/stage3', exist_ok=True)
run('cp runs/hifigan_smoke/metrics.jsonl docs/evidence/phase7/stage3/ && cp -r runs/hifigan_smoke/samples docs/evidence/phase7/stage3/')
check_artifacts('step 7d')

## 7e. Phase 7: Stage 1 on real speech (normal and frozen codebook)
*Off by default: run 4 recorded it (`docs/evidence/colab_run4/stage1_smoke.json`).*

In [ ]:
RUN_STAGE1 = False  # recorded in run 4
if RUN_STAGE1:
    from xtts_trainer.config import DVAEConfig
    from xtts_trainer.stage1_dvae.train import CodebookCollapseError, train_dvae
    stage1 = {}
    for mode, frozen in (('normal', False), ('frozen_codebook', True)):
        out = f'runs/dvae_smoke_{mode}'
        try:
            best1 = train_dvae('sample_audio/dataset.jsonl', out, DVAEConfig(batch_size=4, epochs=20, val_every_steps=10,
                                                                             frozen_codebook=frozen),
                               base_checkpoint='xtts_base/dvae.pth', device='cuda')
            ck1 = torch.load(best1, map_location='cpu', weights_only=False)
            stage1[mode] = {'outcome': 'completed', 'best': ck1['val'], 'reference': ck1['reference']}
        except CodebookCollapseError as e:
            stage1[mode] = {'outcome': 'codebook_collapse', 'error': str(e)}
        stage1[mode]['metrics'] = [json.loads(l) for l in open(f'{out}/metrics.jsonl')]
    json.dump(stage1, open('docs/evidence/phase7/stage1_smoke.json', 'w'), indent=2)

## 8. Stock baseline across all 5 eval languages (Phase 1), at pilot scale
Per language: 12 sentences × 5 voices × 3 seeds = 180 utterances (900 in all, about an hour on a T4). That's the
scale whose seed noise sized the A/B gates (run 2: about ±0.02 WER and SECS), and a fine-tune's A/B must use the
same `subset_size`. English takes 2 sentences from each of its 6 categories; the other languages have no categories,
so they take their first 12. Set `BASELINE_SUBSET = None` for the full set (about 3 times longer).

Each language is its own manifest, written to Drive as it finishes along with an updated index. On resume, a manifest
already on Drive is reused only if it records exactly the expected inputs (commit, clean tree, code fingerprint, eval-set
hash, locked checkpoint hashes, reference voices, seeds, subset, metric version, language) and matches its SHA-256 in the
existing Drive index; otherwise the cell stops rather than mix inputs or adopt edited results.

In [ ]:
import time
from eval.provenance import expected_baseline_inputs, index_manifest_mismatches, manifest_mismatches, write_baseline_index
BASELINE_SUBSET = 12
SEEDS = [42, 123, 999]
LANGUAGES = ['en', 'es', 'fr', 'de', 'hi']
assert sorted(LANGUAGES) == sorted(['en'] + list(json.load(open('eval/dataset/eval_set.json'))['target_languages']))
check_artifacts('before step 8')
changed = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], text=True).strip()
assert not changed, f'tracked files changed since the clone, so manifests would not match {COMMIT}:\n{changed}'
expected = expected_baseline_inputs(COMMIT, seeds=SEEDS, subset_size=BASELINE_SUBSET)
tag = f's{BASELINE_SUBSET}' if BASELINE_SUBSET else 'full'
index_path = f'eval/baselines/base_xtts_{tag}_index.json'
drive_index_path = f'{RESULTS}/base_xtts_{tag}_index.json'
drive_index = json.load(open(drive_index_path)) if os.path.exists(drive_index_path) else None
os.makedirs('eval/baselines', exist_ok=True)
paths = {}
for lang in LANGUAGES:
    name = f'base_xtts_{tag}_{lang}.json'
    local, on_drive = f'eval/baselines/{name}', f'{RESULTS}/{name}'
    if os.path.exists(on_drive):
        assert drive_index, f'{on_drive} exists on Drive but Drive index {drive_index_path} is missing; cannot verify content integrity'
        problems = manifest_mismatches(json.load(open(on_drive)), expected, lang)
        assert not problems, f'{on_drive} was measured on different inputs; move it aside to re-measure:\n' + '\n'.join(problems)
        index_problems = index_manifest_mismatches(on_drive, drive_index, lang)
        assert not index_problems, f'{on_drive} does not match Drive index {drive_index_path}; move it aside to re-measure:\n' + '\n'.join(index_problems)
        run(f'cp {on_drive} {local}')
        print(lang, 'reused from Drive (inputs and index hash verified)')
    else:
        t0 = time.time()
        run(f'python scripts/run_eval_baseline.py --checkpoint_dir xtts_base --languages {lang} --label base '
            f'--seeds {" ".join(map(str, SEEDS))} ' + (f'--subset_size {BASELINE_SUBSET} ' if BASELINE_SUBSET else '') +
            f'--output {local}', tail=4)
        problems = manifest_mismatches(json.load(open(local)), expected, lang)
        assert not problems, f'{local} does not record the expected inputs:\n' + '\n'.join(problems)
        run(f'cp {local} {on_drive}')
        print(lang, f'{(time.time() - t0) / 60:.0f} min')
    paths[lang] = local
    print(lang, json.dumps(json.load(open(local))['summary_by_language'][lang]))
    drive_index = write_baseline_index(paths, index_path, existing_index=drive_index)
    run(f'cp {index_path} {RESULTS}/')
print(json.dumps(json.load(open(index_path))['files'], indent=1))

## 9. Evidence bundle
Download this zip (also saved to Drive) and bring it back.

In [ ]:
run('zip -qr run5_evidence.zip docs/evidence eval/baselines eval/dataset/reference_voices.lock.json '
    'eval/dataset/eval_set.json eval/reference_voices /content/requirements.lock.txt')
run(f'cp run5_evidence.zip {RESULTS}/')
from google.colab import files
files.download('run5_evidence.zip')